<a href="https://colab.research.google.com/github/ts387/cryostar/blob/main/notebooks/CryoSTAR_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CryoSTAR on Google Colab

[CryoSTAR](https://github.com/bytedance/cryostar) recovers the conformational heterogeneity of a protein complex from cryo-EM particle images. It uses a reference atomic model as a structural prior, so it returns both **an ensemble of structures** and **density maps** for the different conformations ([Li *et al.*, *Nature Methods* 2024](https://doi.org/10.1038/s41592-024-02486-1)).

This notebook installs CryoSTAR, gets a dataset (the official tutorial or your own), trains both stages on a single Colab GPU, and shows the results.

| Step | What happens |
|---|---|
| **1 · Setup** | Check the GPU, optionally connect Google Drive, install CryoSTAR |
| **2 · Data** | Download the tutorial or point to your own particles + reference model, then check them |
| **3 · Stage 1 — atom generator** | Learn a latent code for every particle and a coarse-grained structure for each code |
| **4 · Stage 2 — density generator** | Turn the latent codes into density maps |
| **5 · Save** | Download everything as a `.zip` |

### How to use it
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (any GPU works; a faster one finishes sooner).
2. Run the cells from top to bottom with ▶ or **Shift + Enter**. Each cell has a small form; change the values in the form and then run the cell.
3. Cells marked (optional) can be skipped.

> **Colab machines are temporary.** Everything under `/content` disappears when the runtime disconnects. To keep results, tick *Use Google Drive* in step 1.2 or download them in step 5.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU and load helpers { display-mode: "form" }
#@markdown Run this first. It confirms that a GPU is attached and defines the small helper functions used by the cells below.

import os, re, sys, html, time, math, shutil, subprocess, zipfile
from pathlib import Path
import numpy as np
from IPython.display import display, HTML, Markdown

IN_COLAB = "google.colab" in sys.modules
CONTENT = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = CONTENT / "cryostar"
STAR_DIR = REPO_DIR / "projects" / "star"
DATA_ROOT = CONTENT / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Colours for plots (blue = data series, a light-to-dark blue ramp for densities).
SERIES_BLUE = "#2a78d6"
SERIES_ORANGE = "#eb6834"
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

_NOTE_STYLE = {"info": ("#2a78d6", "ℹ️"), "ok": ("#0ca30c", "✅"),
               "warn": ("#fab219", "⚠️"), "error": ("#d03b3b", "❌")}


def note(msg, kind="info"):
    """Show a short coloured message box."""
    color, icon = _NOTE_STYLE[kind]
    display(HTML(f'<div style="border-left:4px solid {color};padding:6px 12px;margin:6px 0;'
                 f'line-height:1.5">{icon} {msg}</div>'))


def need(**names):
    """Stop with a friendly message if a variable from an earlier cell is missing."""
    missing = [cell for name, cell in names.items() if name not in globals()]
    if missing:
        raise RuntimeError("Please run these cells first: " + "; ".join(dict.fromkeys(missing)))


def sh(cmd, cwd=None, quiet=False):
    """Run a shell command, stream its output into the cell and raise if it fails."""
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        out.append(line)
        if not quiet:
            print(line, end="")
    if proc.wait() != 0:
        if quiet:
            print("".join(out[-40:]))
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return "".join(out)


def fmt_time(seconds):
    seconds = int(seconds)
    h, m, s = seconds // 3600, seconds % 3600 // 60, seconds % 60
    return f"{h}h {m:02d}m" if h else f"{m}m {s:02d}s"


def style_axes(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#999999")
    ax.tick_params(colors="#52514e", labelsize=9)
    ax.grid(True, color="#e6e6e6", linewidth=0.6)
    ax.set_axisbelow(True)


def blue_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("blue_ramp", BLUE_RAMP)


# ---------- configs ----------
def write_config(path, base, **sections):
    """Write an mmengine config that inherits `base` and overrides only the given keys."""
    def lit(v):
        return repr(str(v)) if isinstance(v, Path) else repr(v)
    lines = [f'_base_ = "./{base}"', ""]
    for name, value in sections.items():
        if isinstance(value, dict):
            body = "".join(f"    {k}={lit(v)},\n" for k, v in value.items())
            lines.append(f"{name} = dict(\n{body})")
        else:
            lines.append(f"{name} = {lit(value)}")
    Path(path).write_text("\n".join(lines) + "\n")
    return Path(path)


# ---------- training runs ----------
STEP_RE = re.compile(r"epoch (\d+) \[(\d+)/(\d+)\]")
WARNING_RE = re.compile(r"^\S+\.py:\d+: |^\[(rank\d+\]:\[)?W\d{4}|💡 Tip|litlogger")
TIMESTAMP_RE = re.compile(r"^\d\d/\d\d \d\d:\d\d:\d\d - ")
ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")


def ensure_work_dirs():
    """CryoSTAR writes to projects/star/work_dirs; point it at Google Drive when enabled."""
    link = STAR_DIR / "work_dirs"
    persist = globals().get("PERSIST_DIR")
    if persist is None:
        if not link.exists():
            link.mkdir()
        return link
    target = Path(persist) / "work_dirs"
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if Path(os.readlink(link)) == target:
            return link
        link.unlink()
    elif link.exists():  # runs made before Drive was connected: move them over
        for p in link.iterdir():
            shutil.move(str(p), str(target / p.name))
        link.rmdir()
    link.symlink_to(target, target_is_directory=True)
    return link


def _progress_html(title, step, total, rate, last):
    if not total:
        bar, text = '<progress style="width:100%;height:14px"></progress>', "starting…"
    else:
        step = min(step, total)
        bar = f'<progress value="{step}" max="{total}" style="width:100%;height:14px"></progress>'
        text = f"step {step:,} / {total:,} ({100 * step / total:.1f}%)"
        if rate:
            text += f" · about {fmt_time((total - step) / rate)} left"
    return (f'<div style="margin:4px 0"><b>{title}</b> — {text}<br>{bar}'
            f'<div style="font-family:monospace;font-size:12px;opacity:.75">{last}</div></div>')


def run_stage(title, script, config_path, max_steps=None, max_epochs=None):
    """Run a CryoSTAR training script, stream its log and show a progress bar. Returns the run folder."""
    work = ensure_work_dirs()
    before = set(os.listdir(work))
    rel_cfg = Path(config_path).relative_to(STAR_DIR)
    print(f"$ cd {STAR_DIR} && python {script} {rel_cfg}\n")
    status = display(HTML(_progress_html(title, 0, max_steps, None, "")), display_id=True)
    proc = subprocess.Popen([sys.executable, script, str(rel_cfg)], cwd=STAR_DIR,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines, hidden, bad_images, shown = [], 0, 0, 0
    skip_next = collapsing = False
    total, first, rc = max_steps, None, None
    t_start = time.time()
    try:
        for line in proc.stdout:
            lines.append(line)
            plain = ANSI_RE.sub("", line)
            message = plain.split("INFO - ", 1)[-1].strip()
            m = STEP_RE.search(plain)
            if m:
                epoch, batch, n_batches = map(int, m.groups())
                if total is None and max_epochs:
                    total = max_epochs * n_batches
                step = epoch * n_batches + batch
                first = first or (step, time.time())
                elapsed = time.time() - first[1]
                rate = (step - first[0]) / elapsed if elapsed > 0 and step > first[0] else None
                status.update(HTML(_progress_html(title, step, total, rate, html.escape(message))))
            if "Particle image" in plain and "invalid" in plain:
                bad_images += 1
            # Keep the cell readable: fold the full config / model printouts and library warnings.
            if collapsing:
                if not (TIMESTAMP_RE.match(plain) or plain.startswith("Traceback") or "Error" in plain):
                    hidden += 1
                    continue
                collapsing = False
            if WARNING_RE.search(plain):
                hidden, skip_next = hidden + 1, True
                continue
            if skip_next and plain[:1].isspace():
                hidden += 1
                continue
            skip_next = False
            if TIMESTAMP_RE.match(plain) and (message in ("Config:", "Model summary:") or message.startswith("CryoModel(")):
                collapsing = True
                line = line.rstrip("\n").rstrip("(") + " … (folded, see stdout.txt)\n"
            print(line, end="")
            shown = len(lines)
        rc = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
    new = sorted(set(os.listdir(work)) - before)
    run_dir = (work / new[-1]) if new else None
    if run_dir is not None:
        (run_dir / "stdout.txt").write_text("".join(lines))
    if hidden:
        print(f"\n({hidden} lines of warnings and printouts folded; the full output is in {run_dir}/stdout.txt)")
    if bad_images:
        note(f"{bad_images} particle image(s) could not be read and were replaced by zeros. "
             "Check PARTICLES_ROOT in step 2.1.", "warn")
    if rc == 0:
        status.update(HTML(_progress_html(title, total or 1, total or 1, None,
                                          f"finished in {fmt_time(time.time() - t_start)}")))
        return run_dir
    if rc is None:
        note("Training stopped. Results from evaluations that already finished are still in "
             f"<code>{run_dir}</code>.", "warn")
        return run_dir
    log = ANSI_RE.sub("", "".join(lines[-200:]))
    if shown < len(lines):  # the end of the output was folded away: show it now
        print("\n──── last lines of the output ────\n" + "".join(log.splitlines(keepends=True)[-30:]))
    hint = ""
    if "out of memory" in log.lower():
        hint = "The GPU ran out of memory: lower BATCH_SIZE or IMAGE_SIZE and run again. "
    elif "No such file or directory" in log or "FileNotFoundError" in log:
        hint = "A file could not be found: re-check the paths in step 2. "
    raise RuntimeError(f"Training failed (exit code {rc}). {hint}See the log above"
                       + (f" or {run_dir}/stdout.txt." if run_dir else "."))


def result_dirs(run_dir, marker):
    """Evaluation sub-folders (named <epoch>_<step>) of a run that contain `marker`, oldest first."""
    run_dir = Path(run_dir)
    return sorted(p for p in run_dir.iterdir()
                  if p.is_dir() and re.fullmatch(r"\d{4}_\d{7}", p.name) and (p / marker).exists())


def read_metrics(run_dir):
    """Parse the loss lines of a CryoSTAR log into {metric: ([steps], [values])}."""
    metrics = {}
    for log in sorted(Path(run_dir).glob("*.log")):
        for line in log.read_text().splitlines():
            m = STEP_RE.search(line)
            if not m or " | " not in line:
                continue
            epoch, batch, n_batches = map(int, m.groups())
            step = epoch * n_batches + batch
            for part in line.split(" | ")[1:]:
                key, _, value = part.partition(": ")
                try:
                    metrics.setdefault(key.strip(), ([], []))
                    metrics[key.strip()][1].append(float(value))
                    metrics[key.strip()][0].append(step)
                except ValueError:
                    pass
    return metrics


def split_models(pdb_text):
    """Split a multi-model PDB into a list of single-model PDB strings."""
    models, current = [], []
    for line in pdb_text.splitlines():
        if line.startswith("MODEL"):
            current = []
        elif line.startswith("ENDMDL"):
            models.append("\n".join(current) + "\nEND\n")
        elif line.startswith(("ATOM", "HETATM")):
            current.append(line)
    return models or [pdb_text]


# ---------- cryoDRGN inputs -> RELION .star ----------
# pose.pkl holds (rots, trans). rots[i] is RELION's Euler_angles2matrix(rot, tilt, psi), which is also the
# matrix CryoSTAR builds from a .star. trans is the RELION origin shift as a fraction of the box. ctf.pkl is
# (N, 9): D, Apix (of the original stack), dfU, dfV (Å), dfang (°), kV, Cs (mm), amplitude contrast, phase
# shift (°). This holds for pkls from parse_pose_star / parse_ctf_star and from parse_pose_csparc /
# parse_ctf_csparc alike.

class RowCountError(ValueError):
    pass


def _load_pkl(path):
    import pickle
    with open(path, "rb") as f:
        return pickle.load(f)


def _as_array(x):
    if hasattr(x, "detach"):  # torch tensor
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float64)


def _stack_files(particles):
    """Files behind a .mrcs/.mrc stack or a cryoDRGN .txt list, with image counts, box size and header Å/px."""
    import mrcfile
    p = Path(particles)
    if p.name.lower().endswith((".ft.mrcs", ".ft.txt")):
        raise ValueError(f"{p.name} was written by `cryodrgn preprocess` and holds Fourier-space images. "
                         "Use the original particle stack, or one made with `cryodrgn downsample`.")
    if p.suffix.lower() in (".mrcs", ".mrc"):
        files = [p]
    elif p.suffix.lower() == ".txt":
        lines = [l.strip() for l in p.read_text().splitlines() if l.strip()]
        files = [Path(l) if os.path.isabs(l) else p.parent / l for l in lines]
        if not files:
            raise ValueError(f"{p} lists no particle stacks.")
    elif p.suffix.lower() in (".star", ".cs"):
        raise ValueError(f"{p.name}: give the consolidated particle stack (.mrcs, or the .txt written by "
                         "`cryodrgn downsample --chunk`), not the .star/.cs file.")
    else:
        raise ValueError(f"{p.name}: the particle stack must be a .mrcs/.mrc file or a cryoDRGN .txt list.")
    counts, sizes, hdr_apix = [], set(), None
    for f in files:
        if not f.is_file():
            raise FileNotFoundError(f"Particle stack not found: {f}")
        with mrcfile.open(f, mode="r", header_only=True, permissive=True) as m:
            sizes.add((int(m.header.nx), int(m.header.ny)))
            counts.append(max(int(m.header.nz), 1))
            hdr_apix = float(m.voxel_size.x) if hdr_apix is None else hdr_apix
    (nx, ny), = sizes if len(sizes) == 1 else [(0, 1)]
    if len(sizes) != 1 or nx != ny:
        raise ValueError(f"Particle images must be square and all the same size; found {sorted(sizes)}.")
    if nx % 2:
        raise ValueError(f"The particles are {nx} px, an odd size. Old cryoDRGN `preprocess` wrote Fourier-space "
                         "stacks of D+1 px, which can't be used; use the original stack or `cryodrgn downsample` "
                         "output (even box sizes).")
    return files, np.asarray(counts), nx, hdr_apix


def _match_rows(n_stack, n_pose, n_ctf, ind):
    """Rows of the stack, pose.pkl and ctf.pkl that describe the same particles (cryoDRGN's --ind rules)."""
    lens = {"particle stack": n_stack, "pose.pkl": n_pose, "ctf.pkl": n_ctf}
    listing = ", ".join(f"{k} {v:,}" for k, v in lens.items())
    if ind is None:
        if n_stack == n_pose == n_ctf:
            r = np.arange(n_stack)
            return r, r, r, f"stack, pose.pkl and ctf.pkl all have {n_stack:,} rows"
        raise RowCountError(f"The inputs have different numbers of particles ({listing}). If cryoDRGN selected "
                            "particles with an ind.pkl (--ind), give it as IND_PKL.")
    n, n_full = len(ind), max(lens.values())
    if n_full == n:
        if any(v != n for v in lens.values()):
            raise RowCountError(f"The row counts don't fit ind.pkl ({n:,} entries): {listing}.")
        if np.array_equal(ind, np.arange(n)) or ind.max() >= n:
            r = np.arange(n)
            return r, r, r, ("ind.pkl selects every row" if ind.max() < n
                             else f"stack and pkls were already filtered with ind.pkl ({n:,} rows)")
        raise RowCountError(f"Everything has {n:,} rows and ind.pkl only reorders them, so it is unclear whether "
                            "the inputs are already in ind.pkl order. Leave IND_PKL empty if they are.")
    if ind.max() >= n_full:
        raise RowCountError(f"ind.pkl refers to particle {ind.max():,} but the largest input has {n_full:,} rows.")
    rows, parts = {}, []
    for k, v in lens.items():
        if v == n_full:
            rows[k] = ind
            parts.append(f"{k}: all {v:,} rows, selected with ind.pkl")
        elif v == n:
            rows[k] = np.arange(n)
            parts.append(f"{k}: already filtered ({v:,} rows)")
        else:
            raise RowCountError(f"With ind.pkl ({n:,} entries) each input needs {n_full:,} rows (all particles) "
                                f"or {n:,} rows (already filtered); got {listing}.")
    return rows["particle stack"], rows["pose.pkl"], rows["ctf.pkl"], "; ".join(parts)


def cryodrgn_to_star(particles, pose_pkl, ctf_pkl, out_star, ind_pkl=None, apix=None):
    """Write a RELION 3.1 .star with which CryoSTAR sees the same particles, poses and CTF as cryoDRGN."""
    import warnings
    import pandas as pd, starfile
    from scipy.spatial.transform import Rotation
    notes = []
    pose = _load_pkl(pose_pkl)
    if isinstance(pose, (tuple, list)) and len(pose) == 2:
        rots, trans = _as_array(pose[0]), _as_array(pose[1])
    else:
        rots = _as_array(pose[0] if isinstance(pose, (tuple, list)) else pose)
        trans = np.zeros((len(rots), 2))
        notes.append("pose.pkl holds rotations only, so all shifts are set to 0.")
    if rots.ndim != 3 or rots.shape[1:] != (3, 3) or trans.shape != (len(rots), 2):
        raise ValueError(f"{Path(pose_pkl).name} doesn't look like a cryoDRGN pose.pkl "
                         f"(rotations {rots.shape}, shifts {trans.shape}).")
    ctf = _as_array(_load_pkl(ctf_pkl))
    if ctf.ndim != 2 or ctf.shape[1] != 9:
        old = " It looks like the 8-column layout of cryoDRGN < 0.1.1; re-run parse_ctf_*." \
            if ctf.ndim == 2 and ctf.shape[1] == 8 else ""
        raise ValueError(f"{Path(ctf_pkl).name} should be an (N, 9) cryoDRGN ctf.pkl, not {ctf.shape}.{old}")
    ind = None
    if ind_pkl:
        ind = np.asarray(_load_pkl(ind_pkl)).reshape(-1)
        ind = (np.flatnonzero(ind) if ind.dtype == bool else ind).astype(np.int64)
    files, counts, box, hdr_apix = _stack_files(particles)
    stack_rows, pose_rows, ctf_rows, rows_desc = _match_rows(int(counts.sum()), len(rots), len(ctf), ind)
    rots, trans, ctf = rots[pose_rows], trans[pose_rows], ctf[ctf_rows]
    n = len(stack_rows)

    bad = np.flatnonzero(~(np.isfinite(rots).all(axis=(1, 2)) & np.isfinite(trans).all(axis=1)))
    if len(bad):
        raise ValueError(f"{len(bad):,} particles have NaN poses (e.g. rows {bad[:5].tolist()}). cryoDRGN's "
                         "parse_pose_csparc gives NaN when a cryoSPARC pose is exactly zero; remove those "
                         "particles (cryodrgn_utils filter_pkl and an ind.pkl) and convert again.")
    if (np.abs(np.linalg.det(rots) - 1).max() > 1e-3
            or np.abs(rots @ rots.transpose(0, 2, 1) - np.eye(3)).max() > 1e-3):
        raise ValueError("pose.pkl contains matrices that are not rotations.")
    tmax = float(np.abs(trans).max()) if n else 0.0
    if tmax > 1:
        raise ValueError(f"pose.pkl shifts reach {tmax:.2f} box lengths, so they are in pixels rather than "
                         "fractions of the box (as `cryodrgn abinit` writes them). Divide them by the box size.")
    if not np.isfinite(ctf[:, :8]).all():
        raise ValueError("ctf.pkl has missing (NaN) defocus, voltage, Cs or amplitude-contrast values.")

    D0, apix0 = float(ctf[0, 0]), float(ctf[0, 1])
    phys = ctf[:, 0] * ctf[:, 1]  # box size in Å; unchanged by downsampling
    if apix:
        apix = float(apix)
        if abs(apix * box / phys[0] - 1) > 1e-3:
            notes.append(f"PIXEL_SIZE {apix:g} Å × {box} px = {apix * box:.1f} Å, but ctf.pkl implies a "
                         f"{phys[0]:.1f} Å box ({D0:g} px × {apix0:g} Å). Using PIXEL_SIZE.")
    else:
        if np.ptp(phys) > 1e-5 * phys[0]:
            raise ValueError("ctf.pkl mixes particles with different pixel or box sizes (several optics groups); "
                             "CryoSTAR supports one. Enter PIXEL_SIZE to use a single value.")
        apix = float(phys[0]) / box
    if hdr_apix and abs(hdr_apix / apix - 1) > 0.01:
        notes.append(f"The stack's MRC header says {hdr_apix:.4f} Å/px; {apix:.4f} Å/px from ctf.pkl is used "
                     f"({D0:g} px × {apix0:g} Å re-boxed to {box} px). `cryodrgn downsample` writes unreliable "
                     "header pixel sizes for .cs/.txt inputs; if your stack has a correct header, set PIXEL_SIZE.")
    if box < D0:
        notes.append(f"The stack is downsampled from {D0:g} to {box} px. `cryodrgn downsample` scales pixel values "
                     f"by ({D0:g}/{box})² = {(D0 / box) ** 2:.2f}. CryoSTAR's losses ignore this overall scale; "
                     "only stage 1's encoder sees larger inputs. A full-size stack avoids it if you have one.")
        if int(D0) % 2:
            notes.append(f"The original box ({D0:g} px) is odd: `cryodrgn downsample` then shifts images by one "
                         "original pixel relative to the poses.")
    consts = {}
    for col, key, label in ((5, "rlnVoltage", "voltage"), (6, "rlnSphericalAberration", "Cs"),
                            (7, "rlnAmplitudeContrast", "amplitude contrast")):
        values, freq = np.unique(np.round(ctf[:, col], 6), return_counts=True)
        consts[key] = float(values[np.argmax(freq)])
        if len(values) > 1:
            notes.append(f"The {label} varies between particles ({len(values)} values); CryoSTAR uses one, "
                         f"so the most common, {consts[key]:g}, is written.")
    phase = np.nan_to_num(ctf[:, 8])
    if np.any(phase != 0):
        notes.append(f"{np.count_nonzero(phase):,} particles have a phase shift (up to {np.abs(phase).max():.0f}°). "
                     "CryoSTAR's CTF ignores phase shifts, so phase-plate data won't be modelled correctly.")

    # Image names are relative to the folder holding all stacks, so that folder's own path (which may contain
    # '@', quotes or symlinks on Google Drive) never appears in the .star; dataset_dir points at it instead.
    real = [Path(os.path.realpath(f)) for f in files]
    root = Path(os.path.commonpath([str(r.parent) for r in real]))
    rel = [str(r.relative_to(root)) for r in real]
    for name in rel:
        if any(ch in name for ch in "@'\""):
            raise ValueError(f"The stack file name '{name}' contains '@' or a quote, which a .star file can't "
                             "hold. Rename the file (and its line in the .txt list).")
    file_of_row = np.repeat(np.arange(len(files)), counts)
    index_in_file = np.concatenate([np.arange(c) for c in counts])
    names = [f"{index_in_file[r] + 1:06d}@{rel[file_of_row[r]]}" for r in stack_rows]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # gimbal lock at tilt 0/180: angles differ, the matrix doesn't
        euler = Rotation.from_matrix(rots.transpose(0, 2, 1)).as_euler("ZYZ", degrees=True)
    origin_A = trans * box * apix  # RELION origin (x, y) in Å
    particles_df = pd.DataFrame({
        "rlnImageName": names, "rlnAngleRot": euler[:, 0], "rlnAngleTilt": euler[:, 1], "rlnAnglePsi": euler[:, 2],
        "rlnOriginXAngst": origin_A[:, 0], "rlnOriginYAngst": origin_A[:, 1],
        "rlnDefocusU": ctf[:, 2], "rlnDefocusV": ctf[:, 3], "rlnDefocusAngle": ctf[:, 4],
        "rlnPhaseShift": phase, "rlnOpticsGroup": np.ones(n, dtype=int)})
    optics_df = pd.DataFrame({
        "rlnOpticsGroupName": ["opticsGroup1"], "rlnOpticsGroup": [1], **{k: [v] for k, v in consts.items()},
        "rlnImagePixelSize": [apix], "rlnImageSize": [box], "rlnImageDimensionality": [2]})
    out_star = Path(out_star)
    out_star.parent.mkdir(parents=True, exist_ok=True)
    starfile.write({"optics": optics_df, "particles": particles_df}, out_star, overwrite=True, float_format="%.8f")

    back = starfile.read(out_star, always_dict=True)["particles"]
    angles = back[["rlnAngleRot", "rlnAngleTilt", "rlnAnglePsi"]].to_numpy(float)
    again = Rotation.from_euler("ZYZ", angles, degrees=True).as_matrix().transpose(0, 2, 1)
    if len(back) != n or (n and np.abs(again - rots).max() > 1e-5):
        raise RuntimeError("The written .star does not reproduce pose.pkl; please report this.")
    return dict(star=out_star, root=root, n=n, box=box, apix=apix, D0=D0, apix0=apix0, rows=rows_desc,
                files=files, consts=consts, notes=notes, rots=rots, origin_A=origin_A)


def find_cryodrgn_inputs(folder="", particles="", pose="", ctf="", ind=""):
    """Pick the stack and pkls, preferring the ones named in cryoDRGN's config.yaml when there is a choice."""
    import mrcfile, yaml
    folder = Path(folder) if folder else None
    runs, stacks, pkls = [], [], []
    if folder is not None:
        if not folder.is_dir():
            raise FileNotFoundError(f"{folder} is not a folder.")
        for cfg in sorted({*folder.glob("config.yaml"), *folder.glob("*/config.yaml"), *folder.glob("*/*/config.yaml")}):
            try:
                c = yaml.safe_load(cfg.read_text()) or {}
            except Exception:
                continue
            ds = c.get("dataset_args") or {}
            if ds.get("particles"):
                runs.append(dict(config=cfg, version=c.get("version"),
                                 **{k: ds.get(k) for k in ("particles", "poses", "ctf", "ind", "invert_data")}))
        listed = set()
        for t in sorted(folder.glob("*.txt")):
            lines = [l.strip() for l in t.read_text(errors="ignore").splitlines() if l.strip()]
            if lines and all(l.lower().endswith((".mrcs", ".mrc")) for l in lines) and not t.name.endswith(".ft.txt"):
                stacks.append(t)
                listed |= {Path(l).name for l in lines}
        for f in sorted([*folder.glob("*.mrcs"), *folder.glob("*.mrc")]):
            if f.name in listed or f.name.endswith(".ft.mrcs"):
                continue
            try:
                with mrcfile.open(f, mode="r", header_only=True, permissive=True) as m:
                    if f.suffix.lower() == ".mrc" and int(m.header.nz) == int(m.header.nx):
                        continue  # a cubic .mrc is a map, not a particle stack
            except Exception:
                continue
            stacks.append(f)
        pkls = sorted(folder.glob("*.pkl"))
    used = {k: {Path(str(r[k])).name for r in runs if r.get(k)} for k in ("particles", "poses", "ctf", "ind")}

    def pick(given, candidates, key, what):
        if given:
            return Path(given)
        if folder is None:
            raise ValueError(f"Give the {what} or CRYODRGN_FOLDER.")
        if len(candidates) == 1:
            return candidates[0]
        preferred = [c for c in candidates if c.name in used[key]]
        if len(preferred) == 1:
            return preferred[0]
        if not candidates:
            raise FileNotFoundError(f"No {what} found in {folder}; fill in its path.")
        raise ValueError(f"Several possible {what} files in {folder}: " + ", ".join(c.name for c in candidates)
                         + ". Fill in the one to use.")

    chosen = dict(particles=pick(particles, stacks, "particles", "particle stack"),
                  pose_pkl=pick(pose, [p for p in pkls if "pose" in p.name.lower()], "poses", "pose .pkl"),
                  ctf_pkl=pick(ctf, [p for p in pkls if "ctf" in p.name.lower()], "ctf", "ctf .pkl"),
                  ind_pkl=Path(ind) if ind else None)
    ind_options = [p for p in pkls if "ind" in p.name.lower()]
    ind_options = [p for p in ind_options if p.name in used["ind"]] or ind_options
    return chosen, runs, ind_options


# ---------- the GPU check itself ----------
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
HAS_GPU = gpu.returncode == 0 and bool(gpu.stdout.strip())
if HAS_GPU:
    note(f"GPU: <b>{gpu.stdout.strip()}</b>", "ok")
else:
    note("No GPU found. CryoSTAR needs one: choose <b>Runtime ▸ Change runtime type ▸ T4 GPU</b>, "
         "then run this cell again.", "error")
mem_gb = int(open("/proc/meminfo").readline().split()[1]) / 1e6
print(f"CPU cores: {os.cpu_count()} · RAM: {mem_gb:.0f} GB · "
      f"free disk: {shutil.disk_usage(CONTENT).free / 1e9:.0f} GB")

In [ ]:
#@title 1.2 · (optional) Keep results on Google Drive { display-mode: "form" }
#@markdown Tick the box to write training outputs (and the tutorial download) to your Google Drive, so a disconnect never loses finished work. Colab will ask for permission to access Drive.
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "CryoSTAR"  #@param {type:"string"}
#@markdown Outputs go to `MyDrive/<DRIVE_FOLDER>/work_dirs`. Writing to Drive is a bit slower than the local disk.

assert "need" in globals(), "Please run cell 1.1 first."
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive can only be mounted inside Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    note(f"Results will be saved in <code>{PERSIST_DIR}</code>.", "ok")
else:
    PERSIST_DIR = None
    note("Results stay on this Colab machine. Download them in step 5 before you close the tab.")

In [ ]:
#@title 1.3 · Install CryoSTAR { display-mode: "form" }
#@markdown Clones the code and installs the packages Colab doesn't already have (1–2 minutes). Colab's own PyTorch and NumPy are kept.
REPO_URL = "https://github.com/ts387/cryostar"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

assert "need" in globals(), "Please run cell 1.1 first."
if not (REPO_DIR / ".git").exists():
    sh(f"git clone --quiet --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    print(f"Using the existing copy in {REPO_DIR}")

# Compatibility fixes for current packages. Each one is skipped when the code already has it:
# biotite>=1.0 renamed PDBxFile to CIFFile, and recent NumPy releases removed np.row_stack.
PATCHES = {
    "cryostar/utils/pdb_tools.py": [(
        "from biotite.structure.io.pdbx import PDBxFile, get_structure\n",
        "from biotite.structure.io.pdbx import get_structure\n"
        "try:\n"
        "    from biotite.structure.io.pdbx import PDBxFile\n"
        "except ImportError:\n"
        "    # biotite>=1.0 replaced PDBxFile with CIFFile\n"
        "    from biotite.structure.io.pdbx import CIFFile as PDBxFile\n")],
    "cryostar/utils/dist_loss.py": [("np.row_stack(", "np.vstack(")],
}
for rel, edits in PATCHES.items():
    path = REPO_DIR / rel
    src = path.read_text()
    for old, new in edits:
        if old in src:
            src = src.replace(old, new)
            print(f"Applied compatibility fix to {rel}")
    path.write_text(src)

print("Installing packages…")
sh(f"{sys.executable} -m pip install -q lightning mmengine einops torchinfo umap-learn 'biotite>=1.0' "
   "biopython gemmi starfile mrcfile dm-tree tensorboard py3Dmol plotly scikit-image", quiet=True)
sh(f"{sys.executable} -m pip install -q --no-deps -e {REPO_DIR}", quiet=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))  # lets this notebook import cryostar without a restart

versions = sh(f'{sys.executable} -c "import torch, lightning, mmengine, biotite, numpy, train_atom, train_density; '
              'print(torch.__version__, torch.cuda.is_available(), lightning.__version__, '
              'mmengine.__version__, biotite.__version__, numpy.__version__)"', cwd=STAR_DIR, quiet=True)
torch_v, cuda_ok, lightning_v, mmengine_v, biotite_v, numpy_v = versions.split()[-6:]
print(f"torch {torch_v} (CUDA available: {cuda_ok}) · lightning {lightning_v} · mmengine {mmengine_v} · "
      f"biotite {biotite_v} · numpy {numpy_v}")
note("CryoSTAR is installed.", "ok")

## 2 · Data

CryoSTAR needs three things, all from a **consensus refinement** of your particles:

* **particle images** (`.mrcs` stacks),
* each particle's **pose and CTF**: either a **`.star` file** (`rlnAngleRot/Tilt/Psi`, `rlnOriginX/YAngst`, defocus) from RELION, or from cryoSPARC via `csparc2star.py`; or cryoDRGN's **`pose.pkl` + `ctf.pkl`** (from `cryodrgn parse_pose_star/csparc` and `parse_ctf_star/csparc`), which 2.1 converts to a `.star` for you,
* a **reference model** (`.pdb`/`.cif`) docked into the consensus map, in coordinates whose origin is the centre of the box (2.2 checks this and can fix it).

The tutorial dataset already satisfies all three: 50 interpolated adenylate-kinase conformations, simulated as noisy particles with CTF.

In [ ]:
#@title 2.1 · Choose a dataset { display-mode: "form" }
DATA_SOURCE = "Tutorial: adenylate kinase (1AKE, simulated, 3.4 GB download)"  #@param ["Tutorial: adenylate kinase (1AKE, simulated, 3.4 GB download)", "My own data: a RELION or cryoSPARC .star file", "My own data: cryoDRGN files (particle stack + pose.pkl + ctf.pkl)", "My own data: upload a .zip with a .star file"]
#@markdown ---
#@markdown **Your own data.** Paths may point into Google Drive (e.g. `/content/drive/MyDrive/project/...`); Drive is mounted automatically.
REFERENCE_MODEL = ""  #@param {type:"string"}
#@markdown Atomic model (`.pdb`/`.cif`) docked into the consensus map; 2.2 checks its position.
#@markdown
#@markdown *From a `.star` file:*
STAR_FILE = ""  #@param {type:"string"}
PARTICLES_ROOT = ""  #@param {type:"string"}
#@markdown `PARTICLES_ROOT` is the folder the `rlnImageName` paths are relative to (for RELION, the project folder). Blank = detect it. After uploading a `.zip`, `STAR_FILE` and `REFERENCE_MODEL` can stay blank if it holds one of each.
#@markdown
#@markdown *From cryoDRGN files* (`pose.pkl` / `ctf.pkl` from `cryodrgn parse_pose_*` / `parse_ctf_*`, with the matching particle stack, e.g. from `cryodrgn downsample`). Give the folder and leave the rest blank to find them automatically, or give each file:
CRYODRGN_FOLDER = ""  #@param {type:"string"}
CRYODRGN_PARTICLES = ""  #@param {type:"string"}
POSE_PKL = ""  #@param {type:"string"}
CTF_PKL = ""  #@param {type:"string"}
IND_PKL = ""  #@param {type:"string"}
#@markdown `CRYODRGN_PARTICLES` is a `.mrcs` stack or the `.txt` list written by `cryodrgn downsample --chunk`. `IND_PKL` is only needed if cryoDRGN selected particles with `--ind`.
#@markdown
#@markdown *Both:*
PIXEL_SIZE = 0.0  #@param {type:"number"}
#@markdown Å per pixel of the particle images. `0` = read it from the `.star` file or `ctf.pkl`.
COPY_TO_LOCAL_DISK = True  #@param {type:"boolean"}
#@markdown Copy the particles to Colab's disk first. Training reads images constantly, and reading them from Drive is much slower.

need(sh="1.1 · Check the GPU and load helpers", versions="1.3 · Install CryoSTAR")
import numpy as np, mrcfile, starfile
from cryostar.utils.polymer import Polymer

TUTORIAL_URL = "https://zenodo.org/records/17581921/files/tutorial_data_1ake.zip?download=1"
TUTORIAL_PAGE = "https://zenodo.org/records/17581921"


def find_first(root, patterns):
    for pattern in patterns:
        hits = sorted(Path(root).glob(pattern))
        if hits:
            return hits[0]
    return None


def find_one(root, suffixes, what):
    hits = sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in suffixes and "__MACOSX" not in p.parts)
    if len(hits) != 1:
        listing = "".join(f"\n  {h}" for h in hits[:20]) or " none"
        raise ValueError(f"Found {len(hits)} {what} files in {root}; set the path in the form.{listing}")
    return hits[0]


def read_star(path):
    blocks = starfile.read(path, always_dict=True)
    optics = blocks.get("optics")
    particles = blocks["particles"] if "particles" in blocks else list(blocks.values())[-1]
    return optics, particles


def find_particles_root(star, image_name, user_root=""):
    rel = image_name.split("@")[-1]
    if Path(rel).is_absolute():
        if Path(rel).exists():
            return Path("/")
        raise FileNotFoundError(f"The .star file uses absolute image paths ({rel}) that don't exist here. "
                                "Make them relative to the project folder and set PARTICLES_ROOT.")
    candidates = [Path(user_root)] if user_root else [star.parent, *list(star.parent.parents)[:3]]
    for root in candidates:
        if (root / rel).exists():
            return root
    raise FileNotFoundError(f"Could not find the particle stack '{rel}' next to the .star file or in its parent "
                            "folders. Set PARTICLES_ROOT to the folder it is relative to.")


def maybe_mount_drive(*paths):
    if any(str(p).startswith("/content/drive") for p in paths) and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


CONVERTED = None
if DATA_SOURCE.startswith("Tutorial"):
    tut_dir = DATA_ROOT / "tutorial"
    star = find_first(tut_dir, ["**/uniform_snr0-0001_ctf/simulation.star", "**/simulation.star"])
    if star is None:
        zip_path = (PERSIST_DIR or DATA_ROOT) / "tutorial_data_1ake.zip"
        if not (zip_path.exists() and zipfile.is_zipfile(zip_path)):
            print(f"Downloading the tutorial dataset from {TUTORIAL_PAGE} …")
            !wget -c -q --show-progress -O "{zip_path}.part" "{TUTORIAL_URL}"
            if not zipfile.is_zipfile(f"{zip_path}.part"):
                raise RuntimeError(f"The download failed. Download tutorial_data_1ake.zip from {TUTORIAL_PAGE} "
                                   "yourself and use the upload option or Google Drive instead.")
            os.replace(f"{zip_path}.part", zip_path)
        print("Unpacking…")
        sh(f'unzip -q -o "{zip_path}" -d "{tut_dir}"')
        if PERSIST_DIR is None:
            zip_path.unlink()  # free disk space; it is not needed after unpacking
        star = find_first(tut_dir, ["**/uniform_snr0-0001_ctf/simulation.star", "**/simulation.star"])
    pdb = find_first(tut_dir, ["**/pdbs/1akeA_50.pdb", "**/1akeA_50.pdb"])
    if star is None or pdb is None:
        raise FileNotFoundError(f"Could not find simulation.star and 1akeA_50.pdb under {tut_dir}.")
    root, base_cfg = star.parent, "1ake.py"
else:
    if "cryoDRGN" in DATA_SOURCE:
        if not REFERENCE_MODEL:
            raise ValueError("Fill in REFERENCE_MODEL in the form.")
        maybe_mount_drive(CRYODRGN_FOLDER, CRYODRGN_PARTICLES, POSE_PKL, CTF_PKL, IND_PKL, REFERENCE_MODEL)
        inputs, runs, ind_options = find_cryodrgn_inputs(CRYODRGN_FOLDER, CRYODRGN_PARTICLES, POSE_PKL, CTF_PKL, IND_PKL)
        for run in runs:
            names = {k: Path(str(run[k])).name if run.get(k) else "—" for k in ("particles", "poses", "ctf", "ind")}
            print(f"Found cryoDRGN run {run['config'].parent.name} (cryoDRGN {run['version']}): particles "
                  f"{names['particles']}, poses {names['poses']}, ctf {names['ctf']}, ind {names['ind']}, "
                  f"invert_data {run['invert_data']}")
            if run.get("invert_data") is False and names["particles"] == inputs["particles"].name:
                note(f"Run {run['config'].parent.name} trained on this stack with <code>--uninvert-data</code>, so "
                     "its contrast is the opposite of what CryoSTAR expects. Invert the stack first "
                     "(<code>cryodrgn_utils invert_contrast</code>) and use the inverted copy.", "warn")
        print("Using " + ", ".join(f"{k} {v}" for k, v in inputs.items() if v))
        out_star = DATA_ROOT / "cryodrgn" / (inputs["particles"].stem + "_cryostar.star")
        try:
            CONVERTED = cryodrgn_to_star(**inputs, out_star=out_star, apix=PIXEL_SIZE or None)
        except RowCountError:
            if inputs["ind_pkl"] is not None or len(ind_options) != 1:
                raise
            note(f"The stack and pkls have different lengths, so <code>{ind_options[0].name}</code> is used to "
                 "match them. Give IND_PKL explicitly if that is the wrong file.", "warn")
            inputs["ind_pkl"] = ind_options[0]
            CONVERTED = cryodrgn_to_star(**inputs, out_star=out_star, apix=PIXEL_SIZE or None)
        star, root, pdb = CONVERTED["star"], CONVERTED["root"], Path(REFERENCE_MODEL).expanduser()
        for msg in CONVERTED["notes"]:
            note(msg, "warn")
        print(f"Converted to {star}: {CONVERTED['n']:,} particles ({CONVERTED['rows']}), {CONVERTED['box']} px "
              f"at {CONVERTED['apix']:.4f} Å/px (original {CONVERTED['D0']:g} px at {CONVERTED['apix0']:g} Å/px)")
    elif "upload" in DATA_SOURCE:
        if not IN_COLAB:
            raise RuntimeError("Uploading only works inside Colab; use the path option instead.")
        from google.colab import files
        up_dir = DATA_ROOT / "upload"
        up_dir.mkdir(exist_ok=True)
        print("Choose a .zip with your .star file, particle stacks and reference model…")
        for name in files.upload():
            shutil.move(name, up_dir / name)
            if name.lower().endswith(".zip"):
                sh(f'unzip -q -o "{up_dir / name}" -d "{up_dir}"')
                (up_dir / name).unlink()
        star = Path(STAR_FILE) if STAR_FILE else find_one(up_dir, {".star"}, ".star")
        pdb = Path(REFERENCE_MODEL) if REFERENCE_MODEL else find_one(up_dir, {".pdb", ".cif"}, "model (.pdb/.cif)")
    else:
        if not STAR_FILE or not REFERENCE_MODEL:
            raise ValueError("Fill in STAR_FILE and REFERENCE_MODEL in the form.")
        maybe_mount_drive(STAR_FILE, REFERENCE_MODEL, PARTICLES_ROOT)
        star, pdb = Path(STAR_FILE).expanduser(), Path(REFERENCE_MODEL).expanduser()
    for p in (star, pdb):
        if not p.is_file():
            raise FileNotFoundError(f"{p} does not exist.")
    _, particles = read_star(star)
    if CONVERTED is None:
        root = find_particles_root(star, particles["rlnImageName"].iloc[0], PARTICLES_ROOT)
    base_cfg = "example.py"

    stacks = sorted(set(particles["rlnImageName"].str.split("@").str[-1]))
    local = DATA_ROOT / "own"
    if COPY_TO_LOCAL_DISK and not str(root).startswith(str(DATA_ROOT)) and not Path(stacks[0]).is_absolute():
        size = sum((root / s).stat().st_size for s in stacks)
        free = shutil.disk_usage(CONTENT).free
        if size > free - 5e9:
            note(f"The particles take {size / 1e9:.0f} GB but only {free / 1e9:.0f} GB is free, "
                 "so they will be read from where they are.", "warn")
        else:
            from tqdm.auto import tqdm
            print(f"Copying {len(stacks)} particle stack(s), {size / 1e9:.1f} GB, to the local disk…")
            for s in tqdm(stacks, unit="file"):
                src, dst = root / s, local / s
                if not dst.exists() or dst.stat().st_size != src.stat().st_size:
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(src, dst)
            for f in (star, pdb):
                if f.resolve() != (local / f.name).resolve():
                    shutil.copyfile(f, local / f.name)
            star, pdb, root = local / star.name, local / pdb.name, local

# Read the metadata CryoSTAR needs and check it before any training starts.
optics, particles = read_star(star)
required = ["rlnImageName", "rlnDefocusU", "rlnDefocusV", "rlnDefocusAngle",
            "rlnAngleRot", "rlnAngleTilt", "rlnAnglePsi"]
missing = [c for c in required if c not in particles]
if not ({"rlnOriginXAngst", "rlnOriginYAngst"} <= set(particles) or {"rlnOriginX", "rlnOriginY"} <= set(particles)):
    missing.append("rlnOriginXAngst/rlnOriginYAngst")
for c in ("rlnVoltage", "rlnSphericalAberration", "rlnAmplitudeContrast"):
    if not ((optics is not None and c in optics) or c in particles):
        missing.append(c)
if missing:
    raise ValueError("The .star file is missing columns CryoSTAR needs: " + ", ".join(missing))
if optics is not None and len(optics) > 1:
    raise ValueError(f"The .star file has {len(optics)} optics groups; CryoSTAR supports one. "
                     "Split the particles per optics group (or merge them) first.")

if PIXEL_SIZE > 0:
    apix = float(PIXEL_SIZE)
elif optics is not None and "rlnImagePixelSize" in optics:
    apix = float(optics["rlnImagePixelSize"].iloc[0])
elif {"rlnDetectorPixelSize", "rlnMagnification"} <= set(particles):
    apix = float(particles["rlnDetectorPixelSize"].iloc[0] / particles["rlnMagnification"].iloc[0] * 1e4)
elif base_cfg == "1ake.py":
    apix = 1.0  # set in the tutorial config
else:
    raise ValueError("Could not read the pixel size from the .star file; enter it as PIXEL_SIZE.")
first_stack = root / particles["rlnImageName"].iloc[0].split("@")[-1]
with mrcfile.mmap(first_stack, mode="r", permissive=True) as m:
    box = int(m.data.shape[-1])
model = Polymer.from_pdb(str(pdb))

if CONVERTED is not None:
    # Read a sample of particles back through CryoSTAR's own loader and compare with pose.pkl directly.
    from cryostar.utils.dataio import StarfileDataSet, StarfileDatasetConfig
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        dataset = StarfileDataSet(StarfileDatasetConfig(starfile_path=str(star), dataset_dir=str(root),
                                                        apix=apix, side_shape=box))
    sample = np.unique(np.linspace(0, len(dataset) - 1, min(len(dataset), 200)).astype(int))
    rot_err = shift_err = 0.0
    empty = 0
    for i in sample:
        with contextlib.redirect_stdout(io.StringIO()):
            item = dataset[int(i)]
        empty += int(float(item["proj"].abs().max()) == 0)
        rot_err = max(rot_err, float(np.abs(item["rotmat"].numpy() - CONVERTED["rots"][i]).max()))
        shift = np.array([float(item["shiftX"]), float(item["shiftY"])])
        shift_err = max(shift_err, float(np.abs(shift - CONVERTED["origin_A"][i]).max()))
    if empty or rot_err > 1e-4 or shift_err > 1e-3:
        raise RuntimeError(f"CryoSTAR does not read the converted particles as expected ({empty} unreadable "
                           f"images, rotation difference {rot_err:.2e}, shift difference {shift_err:.2e} Å).")
    note(f"Checked {len(sample)} particles through CryoSTAR's own loader: rotations match pose.pkl to "
         f"{rot_err:.1e} and shifts to {shift_err:.1e} Å, and every image reads.", "ok")

DATA = dict(kind="tutorial" if base_cfg == "1ake.py" else "own", star=star, root=root, pdb=pdb,
            apix=apix, box=box, n=len(particles), base=base_cfg,
            n_aa=int(model.num_amino_acids), n_nt=int(model.num_nucleotides),
            apix_override=PIXEL_SIZE > 0)
display(HTML(f"""<table style="text-align:left">
<tr><th>Particles</th><td>{DATA['n']:,} images of {box}×{box} px at {apix:.3f} Å/px</td></tr>
<tr><th>Reference model</th><td>{DATA['n_aa']:,} amino acids, {DATA['n_nt']:,} nucleotides, {model.num_chains} chain(s)</td></tr>
<tr><th>.star file</th><td><code>{star}</code></td></tr>
<tr><th>Particles root</th><td><code>{root}</code></td></tr>
<tr><th>Model file</th><td><code>{pdb}</code></td></tr></table>"""))
note("Dataset ready.", "ok")

In [ ]:
#@title 2.2 · (recommended) Look at the data and check the model's position { display-mode: "form" }
#@markdown Shows a few particles and the reference model, and checks that the model is centred in the box the way CryoSTAR expects.
#@markdown
#@markdown **Own data:** to compare the model with your consensus map, give its path. If you docked the model in ChimeraX on a map whose origin is at its corner, tick `CENTER_MODEL` to move the model's origin to the box centre (this runs CryoSTAR's `cstar_center_origin`).
CONSENSUS_MAP = ""  #@param {type:"string"}
CENTER_MODEL = False  #@param {type:"boolean"}

need(DATA="2.1 · Choose a dataset")
import numpy as np, mrcfile, matplotlib.pyplot as plt, py3Dmol
from cryostar.utils.polymer import Polymer

_, particles = read_star(DATA["star"])
# A few particles, evenly spread through the dataset.
picks = np.linspace(0, DATA["n"] - 1, 8).astype(int)
fig, axes = plt.subplots(1, len(picks), figsize=(2 * len(picks), 2.3))
for ax, i in zip(axes, picks):
    idx, name = particles["rlnImageName"].iloc[i].split("@")
    with mrcfile.mmap(DATA["root"] / name, mode="r", permissive=True) as m:
        img = m.data[int(idx) - 1] if m.data.ndim == 3 else m.data
        ax.imshow(np.asarray(img), cmap="gray")
    ax.set_title(f"#{i}", fontsize=9)
    ax.axis("off")
fig.suptitle("Particle images (individual particles are noisy; that is expected)", fontsize=10)
plt.tight_layout()
plt.show()

if CONSENSUS_MAP:
    maybe_mount_drive(CONSENSUS_MAP)
    prep = DATA_ROOT / "prep"
    prep.mkdir(exist_ok=True)
    with mrcfile.open(CONSENSUS_MAP, permissive=True) as m:
        cmap_data, vs, origin = m.data.copy(), float(m.voxel_size.x), m.header.origin
    origin = np.array([origin.x, origin.y, origin.z], dtype=float)
    print(f"Consensus map: {cmap_data.shape} voxels, {vs:.3f} Å/voxel, origin {origin} Å")
    if CENTER_MODEL and Path(DATA["pdb"]).stem.endswith("_centered"):
        note("The selected model is already the centred copy, so it was not shifted again.")
    elif CENTER_MODEL:
        if np.abs(origin).max() > 1e-3:
            note("This map's origin is not at its corner, so the model may already be centred. "
                 "Check the overlay below before training.", "warn")
        sh(f'cstar_center_origin "{DATA["pdb"]}" "{CONSENSUS_MAP}"', cwd=prep)
        DATA["pdb"] = prep / (Path(DATA["pdb"]).stem + "_centered.pdb")
        note(f"Centred model written to <code>{DATA['pdb']}</code> and selected for training.", "ok")
    sh(f'cstar_generate_gaussian_density "{DATA["pdb"]}" {cmap_data.shape[-1]} {vs} model_density.mrc',
       cwd=prep, quiet=True)
    with mrcfile.open(prep / "model_density.mrc") as m:
        model_vol = m.data.copy()
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
    for ax, axis, label in zip(axes, (0, 1, 2), ("down z", "down y", "down x")):
        ax.imshow(cmap_data.sum(axis), cmap="gray")
        proj = model_vol.sum(axis)
        ax.contour(proj, levels=[proj.max() * 0.15], colors=[SERIES_ORANGE], linewidths=1.5)
        ax.set_title(f"Projection {label}", fontsize=10)
        ax.axis("off")
    fig.suptitle("Consensus map (grey) with the model's outline (orange): they should overlap", fontsize=10)
    plt.tight_layout()
    plt.show()

# CryoSTAR places the model so that (0, 0, 0) is the centre of the box.
model = Polymer.from_pdb(str(DATA["pdb"]))
centre = model.coord.mean(0)
half_box = DATA["box"] * DATA["apix"] / 2
radius = np.linalg.norm(model.coord - centre, axis=1).max()
print(f"Model centre: ({centre[0]:.1f}, {centre[1]:.1f}, {centre[2]:.1f}) Å; box half-width: {half_box:.1f} Å")
if np.abs(centre).max() > 0.25 * half_box:
    note("The model's centre is far from (0, 0, 0). If it was docked with the map origin at the corner, "
         "give CONSENSUS_MAP, tick CENTER_MODEL and run this cell again.", "warn")
elif np.abs(centre).max() + radius > 1.05 * half_box:
    note("Parts of the model stick out of the box. Check that the pixel size and the docking are right.", "warn")
else:
    note("The model sits inside the box, centred as CryoSTAR expects.", "ok")

view = py3Dmol.view(width=640, height=380)
view.addModel(Path(DATA["pdb"]).read_text(), "pdb" if Path(DATA["pdb"]).suffix == ".pdb" else "cif")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

## 3 · Stage 1 — train the atom generator

A variational auto-encoder reads every particle image and outputs a small **latent code** *z* (8 numbers by default) plus a **deformed copy of the reference model**, coarse-grained to one bead per residue (Cα for protein, C1′ for nucleic acid). The deformations are kept physically sensible by restraints on chain connectivity, an elastic network and clashes, and are fitted by comparing projections of the beads with the particle images.

Results are saved at every evaluation, in folders named `<epoch>_<step>`; the last one is the final result.

In [ ]:
#@title 3.1 · Configure stage 1 { display-mode: "form" }
RUN_NAME = "run1"  #@param {type:"string"}
#@markdown A short name for this experiment; outputs go to `work_dirs/atom_colab_<RUN_NAME>…`.
QUICK_TEST = False  #@param {type:"boolean"}
#@markdown Tick for a short run (a few minutes) that checks everything works end to end. The results will be rough.
#@markdown ---
#@markdown **Training length.** `auto` uses the recommended defaults: 24,000 steps for the tutorial, 96,000 for your own data. Pick a value or type your own.
MAX_STEPS = "auto"  #@param ["auto", "12000", "24000", "48000", "96000"] {allow-input: true}
EVALUATE_EVERY = "auto"  #@param ["auto", "3000", "6000", "12000"] {allow-input: true}
#@markdown Each evaluation saves structures, latent codes and a checkpoint.
#@markdown ---
#@markdown **Advanced** (the defaults work well)
BATCH_SIZE = 64  #@param {type:"integer"}
IMAGE_SIZE = "auto"  #@param ["auto", "64", "96", "128", "160", "192", "256"] {allow-input: true}
#@markdown Particles are downsampled to this box size (pixels). `auto`: the original size up to 256 px, otherwise 128.
LOW_PASS_RESOLUTION = "auto"  #@param ["auto", "4", "6", "8", "10", "15"] {allow-input: true}
#@markdown Å. Images are low-pass filtered to this resolution before being compared with the bead model (`auto`: 4 Å for the tutorial, 10 Å for your own data).
LATENT_DIM = 8  #@param {type:"integer"}

need(DATA="2.1 · Choose a dataset")
from mmengine import Config

base = Config.fromfile(str(STAR_DIR / "atom_configs" / DATA["base"]))
if QUICK_TEST:
    steps, every = 2000, 1000
else:
    steps = int(base.trainer.max_steps) if MAX_STEPS == "auto" else int(MAX_STEPS)
    if EVALUATE_EVERY != "auto":
        every = int(EVALUATE_EVERY)
    elif MAX_STEPS == "auto":
        every = int(base.trainer.val_check_interval)
    else:
        every = max(steps // 4, 1)
every = max(1, min(every, steps))
if steps % every:
    steps = math.ceil(steps / every) * every
    note(f"Training length rounded up to {steps:,} steps so that the final step is evaluated.")

RUN_TAG = re.sub(r"[^A-Za-z0-9_-]+", "_", RUN_NAME.strip()) or "run"
dataset_attr = dict(dataset_dir=DATA["root"], starfile_path=DATA["star"], ref_pdb_path=DATA["pdb"])
if DATA["apix_override"] or DATA["kind"] == "own":
    dataset_attr["apix"] = DATA["apix"]
data_process = {}
if IMAGE_SIZE != "auto":
    data_process["down_side_shape"] = min(int(IMAGE_SIZE), DATA["box"])
if LOW_PASS_RESOLUTION != "auto":
    data_process["low_pass_bandwidth"] = float(LOW_PASS_RESOLUTION)
sections = dict(
    exp_name="",
    dataset_attr=dataset_attr,
    data_loader=dict(train_batch_per_gpu=BATCH_SIZE, workers_per_gpu=min(4, os.cpu_count())),
    trainer=dict(max_steps=steps, val_check_interval=every, devices=1),
)
if data_process:
    sections["data_process"] = data_process
if LATENT_DIM != base.model.model_cfg.latent_dim:
    sections["model"] = dict(model_cfg=dict(latent_dim=LATENT_DIM))
ATOM_STEPS = steps
ATOM_CONFIG = write_config(STAR_DIR / "atom_configs" / f"colab_{RUN_TAG}.py", DATA["base"], **sections)

print(f"Config written to {ATOM_CONFIG} (it inherits atom_configs/{DATA['base']}):\n")
print(ATOM_CONFIG.read_text())
note(f"Stage 1 will train for <b>{steps:,} steps</b> with batch size {BATCH_SIZE}, "
     f"evaluating every {every:,} steps.")

In [ ]:
#@title 3.2 · Train stage 1 { display-mode: "form" }
#@markdown A progress bar with the estimated time left appears below, followed by the training log. You can stop early with ■; results from evaluations that already finished stay usable.
#@markdown
#@markdown To reuse a finished run instead of training (for example after a disconnect), give its folder:
USE_EXISTING_RUN = ""  #@param {type:"string"}

need(ATOM_CONFIG="3.1 · Configure stage 1")
if USE_EXISTING_RUN:
    ATOM_RUN = Path(USE_EXISTING_RUN)
else:
    if not HAS_GPU:
        raise RuntimeError("No GPU: choose Runtime ▸ Change runtime type ▸ T4 GPU, then run the notebook again.")
    ATOM_RUN = run_stage("Stage 1 · atom generator", "train_atom.py", ATOM_CONFIG, max_steps=ATOM_STEPS)

finished = result_dirs(ATOM_RUN, "z.npy")
if not finished:
    raise RuntimeError(f"No evaluation has finished in {ATOM_RUN} yet, so there are no results to use.")
ATOM_RESULT = finished[-1]
note(f"Stage 1 results: <code>{ATOM_RESULT}</code>", "ok")

In [ ]:
#@title 3.3 · Training curves and image check { display-mode: "form" }
#@markdown Top: losses during training. Below: particle images next to projections of the predicted bead models for the same particles. The projections are low-pass filtered and include the CTF, so they look smoother and show ripples, but should match each particle's overall shape and orientation. Blue marks the area outside the mask used by the loss.
need(ATOM_RESULT="3.2 · Train stage 1")
import matplotlib.pyplot as plt

NAMES = {"loss": "Total loss", "cryoem(gmm)": "Image term (−correlation)", "con": "Chain connectivity",
         "dist": "Elastic network", "clash": "Clashes", "kld": "KL divergence (weighted)"}
metrics = {k: v for k, v in read_metrics(ATOM_RUN).items() if k in NAMES and any(v[1])}
if metrics:
    fig, axes = plt.subplots(1, len(metrics), figsize=(3.2 * len(metrics), 2.6), squeeze=False)
    for ax, (key, (x, y)) in zip(axes[0], metrics.items()):
        ax.plot(x, y, color=SERIES_BLUE, linewidth=1.5)
        ax.set_title(NAMES[key], fontsize=10)
        ax.set_xlabel("step", fontsize=9)
        style_axes(ax)
    plt.tight_layout()
    plt.show()
else:
    print("No loss values found in the log yet.")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, name, title in zip(axes, ("input_image.png", "pred_gmm_image.png"),
                           ("Particle images", "Projections of the predicted bead models")):
    ax.imshow(plt.imread(ATOM_RESULT / name))
    ax.set_title(title, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#@title 3.4 · Latent space { display-mode: "form" }
#@markdown Every particle has a latent code *z*. Here they are projected onto their first principal components (PCs); clusters or elongated clouds indicate distinct states or continuous motions. The structures in 3.5 and the maps in section 4 are sampled along PC1–PC3.
RUN_UMAP = False  #@param {type:"boolean"}
#@markdown Also show a UMAP embedding (takes a minute or two for large datasets).

need(ATOM_RESULT="3.2 · Train stage 1")
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

z = np.load(ATOM_RESULT / "z.npy")
pca = PCA(min(z.shape[1], 3)).fit(z)
pc = pca.transform(z)
panels = [(pc[:, 0], pc[:, 1], "PC1", "PC2")]
if pc.shape[1] > 2:
    panels.append((pc[:, 0], pc[:, 2], "PC1", "PC3"))
if RUN_UMAP:
    from cryostar.utils.latent_space_utils import run_umap
    emb, _ = run_umap(z)
    panels.append((emb[:, 0], emb[:, 1], "UMAP 1", "UMAP 2"))

fig, axes = plt.subplots(1, len(panels), figsize=(4.4 * len(panels), 4), squeeze=False)
for ax, (x, y, xl, yl) in zip(axes[0], panels):
    hb = ax.hexbin(x, y, gridsize=60, cmap=blue_cmap(), mincnt=1, bins="log")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    style_axes(ax)
    ax.grid(False)
fig.colorbar(hb, ax=axes[0].tolist(), label="particles per bin (log scale)", shrink=0.85)
plt.show()
ratios = ", ".join(f"PC{i + 1} {r:.0%}" for i, r in enumerate(pca.explained_variance_ratio_))
print(f"{len(z):,} particles, latent dimension {z.shape[1]}. Variance explained: {ratios}.")

In [ ]:
#@title 3.5 · Structures along the principal components { display-mode: "form" }
#@markdown Ten bead models sampled from the 5th to the 95th percentile along a PC, played as a movie. Drag to rotate, scroll to zoom.
TRAJECTORY = "PC1"  #@param ["PC1", "PC2", "PC3", "k-means cluster centres"]
STYLE = "cartoon"  #@param ["cartoon", "spheres"]
SHOW_EXTREMES = True  #@param {type:"boolean"}
#@markdown Also show the first (blue) and last (orange) structure superimposed, to see the range of motion.

need(ATOM_RESULT="3.2 · Train stage 1")
import py3Dmol

fname = {"PC1": "pca-1.pdb", "PC2": "pca-2.pdb", "PC3": "pca-3.pdb",
         "k-means cluster centres": "pred.pdb"}[TRAJECTORY]
pdb_text = (ATOM_RESULT / fname).read_text()
frames = split_models(pdb_text)
look = (lambda color: {"cartoon": {"color": color, "style": "trace", "thickness": 0.8}}) if STYLE == "cartoon" \
    else (lambda color: {"sphere": {"color": color, "radius": 1.6}})

view = py3Dmol.view(width=720, height=460)
view.addModelsAsFrames(pdb_text, "pdb")
view.setStyle(look("spectrum"))
view.animate({"loop": "backAndForth", "interval": 250})
view.zoomTo()
print(f"{TRAJECTORY}: {len(frames)} structures from {ATOM_RESULT / fname}")
view.show()

if SHOW_EXTREMES and len(frames) > 1:
    view = py3Dmol.view(width=720, height=460)
    view.addModel(frames[0], "pdb")
    view.setStyle({"model": 0}, look(SERIES_BLUE))
    view.addModel(frames[-1], "pdb")
    view.setStyle({"model": 1}, look(SERIES_ORANGE))
    view.zoomTo()
    view.show()

## 4 · Stage 2 — train the density generator

Stage 2 keeps the latent code *z* that stage 1 assigned to each particle and trains a neural density model that turns any *z* into a 3D map. At the end it writes maps sampled along PC1–PC3 (`vol_pca_<pc>_<frame>.mrc`) and at k-means centres (`vol_kmeans_<i>.mrc`).

In [ ]:
#@title 4.1 · Configure stage 2 { display-mode: "form" }
QUICK_TEST = False  #@param {type:"boolean"}
#@markdown Tick for a short run (a few minutes) that checks everything works end to end. The maps will be rough.
#@markdown ---
MAX_EPOCHS = "auto"  #@param ["auto", "5", "10", "20"] {allow-input: true}
#@markdown Passes over all particles. `auto`: 5 for the tutorial, 20 for your own data.
#@markdown ---
#@markdown **Advanced** (the defaults work well)
BATCH_SIZE = 16  #@param {type:"integer"}
IMAGE_SIZE = "auto"  #@param ["auto", "64", "96", "128", "160", "192", "256"] {allow-input: true}
#@markdown Box size (pixels) of the output maps. `auto`: 128 px, or the original size if smaller. Larger boxes need much more time and GPU memory.
DECODER_WIDTH = "1024"  #@param ["256", "512", "1024"]
#@markdown Width of the density network. Smaller is faster but less detailed.
Z_FILE = ""  #@param {type:"string"}
#@markdown Latent codes to use. Blank = `z.npy` from stage 1 (3.2).

need(DATA="2.1 · Choose a dataset")
import numpy as np
from mmengine import Config

if Z_FILE:
    z_path = Path(Z_FILE)
else:
    need(ATOM_RESULT="3.2 · Train stage 1")
    z_path = ATOM_RESULT / "z.npy"
z = np.load(z_path, mmap_mode="r")
if len(z) != DATA["n"]:
    raise ValueError(f"{z_path} has codes for {len(z):,} particles but the dataset has {DATA['n']:,}.")

base = Config.fromfile(str(STAR_DIR / "density_configs" / DATA["base"]))
epochs = 1 if QUICK_TEST else (int(base.trainer.max_epochs) if MAX_EPOCHS == "auto" else int(MAX_EPOCHS))
every = min(int(base.trainer.check_val_every_n_epoch or epochs), epochs)
if epochs % every:
    epochs = math.ceil(epochs / every) * every
    note(f"Training length rounded up to {epochs} epochs so that the final epoch is evaluated.")
if IMAGE_SIZE == "auto":
    size = min(base.data_process.down_side_shape or 128, DATA["box"])
else:
    size = min(int(IMAGE_SIZE), DATA["box"])

dataset_attr = dict(dataset_dir=DATA["root"], starfile_path=DATA["star"])
if DATA["apix_override"] or DATA["kind"] == "own":
    dataset_attr["apix"] = DATA["apix"]
trainer = dict(max_epochs=epochs, check_val_every_n_epoch=every, devices=1)
batches = DATA["n"] // BATCH_SIZE
if QUICK_TEST:
    trainer["limit_train_batches"] = batches = min(500, batches)

RUN_TAG = globals().get("RUN_TAG", "run")
DENSITY_EPOCHS = epochs
DENSITY_CONFIG = write_config(
    STAR_DIR / "density_configs" / f"colab_{RUN_TAG}.py", DATA["base"],
    exp_name="",
    dataset_attr=dataset_attr,
    extra_input_data_attr=dict(given_z=z_path),
    data_process=dict(down_side_shape=int(size)),
    data_loader=dict(train_batch_per_gpu=BATCH_SIZE, workers_per_gpu=min(4, os.cpu_count())),
    model=dict(hidden=int(DECODER_WIDTH), z_dim=int(z.shape[1])),
    trainer=trainer,
)
print(f"Config written to {DENSITY_CONFIG} (it inherits density_configs/{DATA['base']}):\n")
print(DENSITY_CONFIG.read_text())
note(f"Stage 2 will train for <b>{epochs} epoch(s)</b> (about {batches * epochs:,} steps) "
     f"on {size}×{size} px images, evaluating every {every} epoch(s).")

In [ ]:
#@title 4.2 · Train stage 2 { display-mode: "form" }
#@markdown To reuse a finished run instead of training, give its folder:
USE_EXISTING_RUN = ""  #@param {type:"string"}

need(DENSITY_CONFIG="4.1 · Configure stage 2")
if USE_EXISTING_RUN:
    DENSITY_RUN = Path(USE_EXISTING_RUN)
else:
    if not HAS_GPU:
        raise RuntimeError("No GPU: choose Runtime ▸ Change runtime type ▸ T4 GPU, then run the notebook again.")
    DENSITY_RUN = run_stage("Stage 2 · density generator", "train_density.py", DENSITY_CONFIG,
                            max_epochs=DENSITY_EPOCHS)

finished = result_dirs(DENSITY_RUN, "vol_pca_1_000.mrc")
if not finished:
    raise RuntimeError(f"No evaluation has finished in {DENSITY_RUN} yet, so there are no maps to show.")
DENSITY_RESULT = finished[-1]
note(f"Stage 2 results: <code>{DENSITY_RESULT}</code>", "ok")

In [ ]:
#@title 4.3 · Maps along a principal component { display-mode: "form" }
#@markdown Projections of the ten maps sampled along one PC (frame 0 → 9), viewed down three axes. Compare columns to see what moves.
PC = "PC1"  #@param ["PC1", "PC2", "PC3"]

need(DENSITY_RESULT="4.2 · Train stage 2")
import numpy as np, mrcfile, matplotlib.pyplot as plt

k = PC[-1]
paths = sorted(DENSITY_RESULT.glob(f"vol_pca_{k}_*.mrc"))
if not paths:
    raise FileNotFoundError(f"No maps for {PC} in {DENSITY_RESULT}.")
vols = [mrcfile.read(p) for p in paths]
fig, axes = plt.subplots(3, len(vols), figsize=(1.45 * len(vols), 4.6))
for row, axis in enumerate((0, 1, 2)):
    projs = [v.sum(axis) for v in vols]
    lo, hi = np.percentile(np.stack(projs), [1, 99.8])
    for col, proj in enumerate(projs):
        ax = axes[row, col]
        ax.imshow(proj, cmap="gray", vmin=lo, vmax=hi)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(str(col), fontsize=9)
    axes[row, 0].set_ylabel(("down z", "down y", "down x")[row], fontsize=9)
fig.suptitle(f"Density maps along {PC} (frame number on top)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
#@title 4.4 · 3D movie of the maps { display-mode: "form" }
#@markdown An isosurface of each map along the chosen PC. Press ▶ or drag the slider to step through the frames; drag the view to rotate.
PC = "PC1"  #@param ["PC1", "PC2", "PC3"]
SURFACE_LEVEL = 1.0  #@param {type:"slider", min:0.3, max:3.0, step:0.1}
#@markdown Threshold, as a multiple of the volume expected from the reference model's mass. Lower it to see weaker density.

need(DENSITY_RESULT="4.2 · Train stage 2")
import numpy as np, mrcfile, plotly.graph_objects as go
from skimage.measure import marching_cubes

paths = sorted(DENSITY_RESULT.glob(f"vol_pca_{PC[-1]}_*.mrc"))
with mrcfile.open(paths[0]) as m:
    vs = float(m.voxel_size.x)
vols = [mrcfile.read(p).astype(np.float32) for p in paths]
bin_by = 2 if vols[0].shape[0] > 100 else 1  # keep the meshes light enough for the browser
if bin_by > 1:
    n = vols[0].shape[0] // bin_by * bin_by
    vols = [v[:n, :n, :n].reshape(n // 2, 2, n // 2, 2, n // 2, 2).mean((1, 3, 5)) for v in vols]
    vs *= bin_by

# Threshold so the surface encloses SURFACE_LEVEL × the expected molecular volume
# (~110 Da per residue, ~330 Da per nucleotide, 1.21 Å^3 per Da).
expected = (DATA["n_aa"] * 110 + DATA["n_nt"] * 330) * 1.21 / vs ** 3
frac = float(np.clip(expected * SURFACE_LEVEL / vols[0].size, 1e-4, 0.5))
level = float(np.quantile(vols[len(vols) // 2], 1 - frac))


def mesh(v):
    if not v.min() < level < v.max():
        return go.Mesh3d(x=[], y=[], z=[])
    verts, faces, _, _ = marching_cubes(v, level)
    xyz = verts[:, ::-1] * vs  # (z, y, x) voxels -> (x, y, z) Å
    return go.Mesh3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                     color=SERIES_BLUE, opacity=1.0, flatshading=False,
                     lighting=dict(ambient=0.45, diffuse=0.8, specular=0.2), hoverinfo="skip")


meshes = [mesh(v) for v in vols]
extent = vols[0].shape[0] * vs
axis = dict(range=[0, extent], visible=False)
steps = [dict(method="animate", label=str(i),
              args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
         for i in range(len(meshes))]
fig = go.Figure(
    data=[meshes[0]],
    frames=[go.Frame(data=[m], name=str(i)) for i, m in enumerate(meshes)],
    layout=go.Layout(
        title=f"Density maps along {PC}", height=560, margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube"),
        updatemenus=[dict(type="buttons", x=0.02, y=0.02, xanchor="left", yanchor="bottom",
                          buttons=[dict(label="▶ Play", method="animate",
                                        args=[None, dict(frame=dict(duration=350, redraw=True),
                                                         fromcurrent=True, mode="immediate")])])],
        sliders=[dict(active=0, steps=steps, x=0.15, len=0.8, y=0.02,
                      currentvalue=dict(prefix="frame "))],
    ),
)
fig.show()

## 5 · Save the results

In [ ]:
#@title 5.1 · Download the results { display-mode: "form" }
#@markdown `final results only` packs the last evaluation of each stage plus its config and logs; `complete run folders` packs everything, including intermediate evaluations (much larger).
WHAT = "final results only"  #@param ["final results only", "complete run folders"]

assert "need" in globals(), "Please run cell 1.1 first."
files_to_pack = []  # (file, folder that its path in the zip is relative to)
for run_name, result_name in (("ATOM_RUN", "ATOM_RESULT"), ("DENSITY_RUN", "DENSITY_RESULT")):
    if run_name not in globals():
        continue
    run = Path(globals()[run_name])
    if WHAT == "complete run folders":
        chosen = [f for f in run.rglob("*") if f.is_file()]
    else:
        chosen = [f for f in run.iterdir() if f.is_file()]
        if result_name in globals():
            chosen += [f for f in Path(globals()[result_name]).rglob("*") if f.is_file()]
    files_to_pack += [(f, run.parent) for f in chosen]
if not files_to_pack:
    raise RuntimeError("Nothing to save yet: train stage 1 (3.2) first.")

zip_path = CONTENT / f"cryostar_{globals().get('RUN_TAG', 'run')}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f, rel_to in files_to_pack:
        zf.write(f, f.relative_to(rel_to))
print(f"Packed {zip_path.name}: {zip_path.stat().st_size / 1e6:.0f} MB")
if PERSIST_DIR is not None:
    note(f"Your runs are also saved in Google Drive under <code>{PERSIST_DIR / 'work_dirs'}</code>.", "ok")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print(f"Saved to {zip_path}")

### Looking at the results in ChimeraX

Inside the `.zip`, `atom_…/<epoch>_<step>/` holds the stage-1 results and `density_…/<epoch>_<step>/` the maps.

| File | What it is |
|---|---|
| `pca-1.pdb`, `pca-2.pdb`, `pca-3.pdb` | 10 bead models each, along PC1–PC3 of the latent space |
| `pred.pdb` | bead models at the k-means cluster centres |
| `z.npy` | the latent code of every particle (particles × latent dimension), in `.star` order |
| `vol_pca_<pc>_<frame>.mrc` | density maps along each PC; `z_pca_<pc>.txt` holds the matching codes |
| `vol_kmeans_<i>.mrc` | density maps at k-means cluster centres |
| `ckpt.pt` | model weights |

In [ChimeraX](https://www.cgl.ucsf.edu/chimerax/):

* bead models as a movie: `open pca-1.pdb coordsets true`, then `coordset slider #1`
* maps as a movie: `open vol_pca_1_*.mrc vseries true`, then `vseries slider #2`

## Troubleshooting

* **`CUDA out of memory`** — lower `BATCH_SIZE` or `IMAGE_SIZE` in 3.1 / 4.1 and run the training cell again.
* **The runtime disconnected during training.** Turn on Google Drive (1.2) before training. After reconnecting, run 1.1 → 1.3, 2.1, then 3.1 and give the stage-1 run folder in `USE_EXISTING_RUN` (3.2); stage 2 can then start from its `z.npy`.
* **`WARNING: Particle image … invalid! Setting to zeros.`** — the image paths in the `.star` file don't resolve. Set `PARTICLES_ROOT` in 2.1 to the folder they are relative to.
* **The bead models look distorted or the projections in 3.3 don't match the particles.** Check that the model is docked in the consensus map and centred (2.2), and that the pixel size is right.
* **cryoDRGN inputs.** The stack's rows must be in the same order as the `.star`/`.cs` the pkls were parsed from (`cryodrgn downsample` keeps that order). With cryoDRGN < 4.3, `parse_pose_csparc` needed `-D`: it must have been the box of the particles in the `.cs` file, not a downsampled size; for a heterogeneous refinement it also needed `--hetrefine`. Poses refined by cryoDRGN itself (`--do-pose-sgd`) work, with shifts about 1/D too small.
* **More than one optics group.** CryoSTAR reads CTF constants from a single optics group; process each group separately.
* **Other settings.** Every option of the original configs can be changed: edit `atom_configs/colab_<RUN_NAME>.py` or `density_configs/colab_<RUN_NAME>.py` under `/content/cryostar/projects/star` (Colab's file browser on the left opens them) and rerun the training cell. The commented `example.py` next to them explains each option. The [CryoSTAR user guide](https://byte-research.gitbook.io/cryostar/) has more detail.

## Citation

If CryoSTAR helps your research, please cite:

> Li Y, Zhou Y, Yuan J, Ye F, Gu Q. CryoSTAR: leveraging structural priors and constraints for cryo-EM heterogeneous reconstruction. *Nature Methods* (2024). [doi:10.1038/s41592-024-02486-1](https://doi.org/10.1038/s41592-024-02486-1)